In [ ]:
import os
import sys
import torch
import selfies as sf
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

from input_output import *

from network.selfies2selfies_netwrok_hs_vae import *
from data_utils import *
sys.path.append('../../')
from fingerprint_analysis.calc_fps_rm import calc_metric_listsmiles2smiles

def smiles2selfies(smiles_list):
    return [sf.decoder(s) for s in smiles_list]

%load_ext autoreload
%autoreload 2

In [2]:
path_init = '/home/rkmvu/Dataset/selfies/zinc/'
# GPU device parameter
gpu_device_id = 0# GPU number for multiple GPUs (pytorch takes default 0 or which is availabe next)
device = torch.device("cuda:"+str(gpu_device_id) if torch.cuda.is_available() else "cpu")

# dataset parameters
train_ratio = 0.5
val_ratio = 0.2
test_ratio = 1 - train_ratio - val_ratio
num_tokens = 58
selfies_max_len = 128
flag_selfies_tokens = True
flag_extra_data = False#True
flag_extra_tokens = True
num_extra_tokens = 105
req_selfies_max_len = 128#150
num_train_samples = 3000000#1500000#3000000#10000#12941195#100000#18000000#10000
num_train_test_samples = 100000#10000#100000#

In [3]:
#read selfies tokens
input_filename = ''.join([
    path_init,
    'tokens_selfies_',
    'train_', str(train_ratio).replace('.', '_'),
    '_val_', str(val_ratio).replace('.', '_'),
    '_test_', str(test_ratio).replace('.', '_'), 
    '_', str(num_tokens),
    '.csv'
])
tokens_selfies = _read_tokens_selfies(input_filename)
if flag_extra_tokens:
    tokens_selfies = list(set(tokens_selfies).union(sf.get_semantic_robust_alphabet()))
dict_selfies_tokens = _get_dict_selfies(tokens_selfies)

#read selfies max length
input_filename = ''.join([
    path_init,
    'max_len_selfies_',
    'train_', str(train_ratio).replace('.', '_'),
    '_val_', str(val_ratio).replace('.', '_'),
    '_test_', str(test_ratio).replace('.', '_'), 
    '_', str(selfies_max_len),
    '.txt'
])
selfies_max_len = _read_max_len_selfies(input_filename)

Reading data from "/home/rkmvu/Dataset/selfies/zinc/tokens_selfies_train_0_5_val_0_2_test_0_3_58.csv"
----------------------------------------------------------------------
Done!


In [4]:
len(tokens_selfies)

87

In [25]:
## VAE_GRU parameters
dims_input_data = (selfies_max_len, len(tokens_selfies)) 
dims_output_data = (selfies_max_len, len(tokens_selfies))
num_kernels = [9, 9, 10]#[25, 21, 17]#[11, 13, 15]#[15, 17, 19]#[11, 13, 15]#
size_kernels = [9, 9, 11]#[13, 11, 9]#[9, 9, 11]#[15, 13, 11]#[9, 9, 11]#
num_fc_layer_encoder = 0
dropout_prob = 0.0
dims_latent = 40#7
act_func = 'relu'
scale_latent_space = 1e-2
num_fc_layer_decoder = 0
gru_hidden_size = 500
num_gru = 4
distribution = 'vmf'
space_transform = False
layer_type = '1dcnn_gru'
weight_init = 'xvr_unifrm'
loss_type = 'bce_kld'#'bce_kld_uniform'#
solver_type = 'adam'
num_epoch = 500
batch_size = 512#256#128
learning_rate = 1e-3
save_result_ateach_epoch = 50
result_savepath = ''.join([
    'cache/selfie2selfies_hs_vae/', 
    'train_', str(train_ratio).replace('.', '_'),
    '_val_', str(val_ratio).replace('.', '_'),
    '_test_', str(test_ratio).replace('.', '_'), 
    '_nt_', str(num_tokens),
    '_ml_', str(selfies_max_len),
    '_slft_', str(flag_selfies_tokens),
    '_exd_'+str(flag_extra_data)+'_ext_'+str(flag_extra_tokens)+'_nt_'+str(num_extra_tokens)+'_rsl_'+str(req_selfies_max_len) if flag_extra_data else '_exd_'+str(flag_extra_data),
    '_nts_', str(num_train_samples),
    '_ntts_', str(num_train_test_samples),
    '/'
]).replace('.','_')
#+++++++++++++++++++++++++++++++++++++++++++++++++++

In [26]:
selfies_hs_vae_model = net_selfies2selfies(
    dims_input_data, 
    dims_output_data, 
    max_string_len=selfies_max_len,
    num_kernels=num_kernels, 
    size_kernels=size_kernels, 
    num_fc_layer_encoder=num_fc_layer_encoder, 
    dropout_prob=dropout_prob, 
    dims_latent=dims_latent, 
    act_func=act_func, 
    scale_latent_space=scale_latent_space, 
    num_fc_layer_decoder=num_fc_layer_decoder, 
    gru_hidden_size=gru_hidden_size, 
    num_gru=num_gru, 
    distribution=distribution,
    space_transform=space_transform,
    layer_type=layer_type, 
    device=device, 
    weight_init=weight_init, 
    loss_type=loss_type, 
    solver_type=solver_type, 
    num_epoch=num_epoch, 
    batch_size=batch_size, 
    learning_rate=learning_rate, 
    save_result_ateach_epoch=save_result_ateach_epoch, 
    result_savepath=result_savepath
)
#+++++++++++++++++++++++++++++++++++++++++++++++++++

selfies_hs_vae_model.load_test_network()# load pre-trained best network


----------------------------------------------------------------------
Network summary
Layer (type:depth-idx)                   Output Shape              Param #
CNN1D_GRU_HSVAE                          [512, 128, 87]            --
├─Sequential: 1-1                        [512, 10, 61]             --
│    └─Conv1d: 2-1                       [512, 9, 79]              10,377
│    └─ReLU: 2-2                         [512, 9, 79]              --
│    └─Conv1d: 2-3                       [512, 9, 71]              738
│    └─ReLU: 2-4                         [512, 9, 71]              --
│    └─Conv1d: 2-5                       [512, 10, 61]             1,000
│    └─ReLU: 2-6                         [512, 10, 61]             --
├─Sequential: 1-2                        [512, 40]                 --
│    └─Linear: 2-7                       [512, 40]                 24,440
├─Sequential: 1-3                        [512, 40]                 --
│    └─Linear: 2-8                       [512, 40]      

## **Test the model for representation**

In [ ]:
def encode(smiles_or_selfies):
    z =  selfies_hs_vae_model.selfies2latent_vector(selfies=smiles_or_selfies, 
                                                   dict_selfies_tokens=dict_selfies_tokens)
    return z

def decode(x_latent):
    x = selfies_hs_vae_model.latent_vector2selfies(x_latent=x_latent, 
                                                dict_selfies_tokens=dict_selfies_tokens)
    return x

def name2selfies(name):
    row = df_temp[df_temp['name'] == name].iloc[0]
    return row['selfies']

def selfies2name(selfies):
    row = df_temp[df_temp['selfies'] == selfies].iloc[0]
    return row['name']

def smi2nneigh(selfies, indices, n_neigh=10):

    idx = selfies_main.index(selfies)
    neigh_idx = indices[idx][1:n_neigh+1]
    nn_selfies = [selfies_main[i] for i in neigh_idx]
    nn_names = [selfies2name(x) for x in nn_selfies]
    
    return {'selfies':nn_selfies, 'name':nn_names}


In [8]:
import selfies as sf

smiles2selfie_fn = lambda x: sf.encoder(x)
smiles_clean_fn = lambda x: ('[' not in x) and ('B' not in x.replace('Br', '')) and ('.' not in x)

df = pd.read_csv('/home/rkmvu/Dataset/selfies/zinc/properties_dtratio_0.001_test_0_3_canon_smiles_selfies_with_descriptors_train_0_5_val_0_2_test_0_3.csv')
mask = df['SMILES'].apply(smiles_clean_fn)
df = df[mask].copy()
df['selfies'] = df['SMILES'].apply(smiles2selfie_fn)
selfies_main = df['selfies'].tolist()
df.head(2)

,SMILES,MolWt,TPSA,EState_VSA1,NHOHCount,MolLogP,fr_COO,nAcid,ATSC1c,ATSC1se,...,fr_halogen,fr_ketone,fr_nitro,fr_nitro_arom,fr_nitroso,fr_phenol,fr_sulfone,AUTOCORR2D_156,nBase,selfies
0,COc1ccc(CN2CCC3CN(C(=O)C(C)C(C)(C)C)C3C2)nn1,346.475,58.56,0.000000,0,2.2001,0,0,-0.330215,0.267548,...,0,0,0,0,0,0,0,1.405,1,[C][O][C][=C][C][=C][Branch2][Ring1][S][C][N][...
1,C#CCOC(C)C(=O)N1CC2(CCCN2C(=O)c2ccc(CO)o2)C1,346.383,83.22,6.103966,1,0.6272,0,0,-0.612666,-0.332817,...,0,0,0,0,0,0,0,1.405,0,[C][#C][C][O][C][Branch1][C][C][C][=Branch1][C...


In [9]:
from data_utils import _check_validity_selfies

z = encode(smiles_or_selfies=selfies_main)
selfies_recon = decode(x_latent=z)

exact = [x==y for x, y in zip(selfies_main, selfies_recon)]
selfiles, check_ids, smiles_decode = _check_validity_selfies(selfiles=selfies_recon)

print('='*80)
print(f'Number of SMILES: {len(selfies_main)}')
print(f'Exact reconstruction: {sum(exact)/z.shape[0]}')
print(f'Valid reconstruction: {sum(check_ids)/z.shape[0]}')
print('='*80)


<===doing selfies reconstruction===>
----------------------------------------------------------------------


 36%|███▌      | 5/14 [00:00<00:00, 18.92it/s]

100%|██████████| 14/14 [00:01<00:00, 12.64it/s]


<===doing selfies reconstruction===>
----------------------------------------------------------------------


100%|██████████| 14/14 [00:00<00:00, 16.84it/s]


<===checking selfies validity===>
----------------------------------------------------------------------


100%|██████████| 6813/6813 [00:00<00:00, 7843.20it/s]

Number of SMILES: 6813
Exact reconstruction: 0.0
Valid reconstruction: 1.0


In [10]:
selfies_main[:5]

['[C][O][C][=C][C][=C][Branch2][Ring1][S][C][N][C][C][C][C][N][Branch1][P][C][=Branch1][C][=O][C][Branch1][C][C][C][Branch1][C][C][Branch1][C][C][C][C][Ring1][N][C][Ring1][S][N][=N][Ring2][Ring1][#Branch1]',
 '[C][#C][C][O][C][Branch1][C][C][C][=Branch1][C][=O][N][C][C][Branch2][Ring1][=Branch1][C][C][C][N][Ring1][Branch1][C][=Branch1][C][=O][C][=C][C][=C][Branch1][Ring1][C][O][O][Ring1][#Branch1][C][Ring1][P]',
 '[C][N][C][=Branch1][C][=O][C][=C][C][=C][Branch1][C][C][C][Branch2][Ring1][=N][N][C][=Branch1][C][=O][N][C][Branch1][C][C][C][=C][C][=C][C][=C][Ring1][=Branch1][O][C][C][=C][C][=C][C][=C][Ring1][=Branch1][=C][Ring2][Ring1][O]',
 '[C][O][C][Branch1][C][C][C][C][N][Branch2][Ring1][=Branch1][C][=Branch1][C][=O][C][=C][C][=C][C][=C][Ring1][=Branch1][S][C][Branch1][C][F][Branch1][C][F][F][C][C][Ring2][Ring1][Ring2]',
 '[C][S][=Branch1][C][=O][=Branch1][C][=O][C][C][C][N][C][=Branch1][C][=O][N][Branch1][#C][C][C][O][C][=C][C][=C][C][Branch1][C][Cl][=C][Ring1][#Branch1][C][Ring1][S]

In [12]:
# z.shape
selfies_recon[:5]

['[#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch3][#Branch

In [13]:
_smiles_temp_org = smiles2selfies(selfies_main[:5])
_smiles_temp = smiles2selfies(selfies_recon[:5])
for i in range(5):
    print(_smiles_temp_org[i])
    print(calc_metric_listsmiles2smiles(smiles=_smiles_temp_org[i], from_list=_smiles_temp[i]))
    print('-'*80)

COC1=CC=C(CN2CCC3CN(C(=O)C(C)C(C)(C)C)C3C2)N=N1
  smiles  Euclidean_metric
0                  27.5318
--------------------------------------------------------------------------------
C#CCOC(C)C(=O)N1CC2(CCCN2C(=O)C3=CC=C(CO)O3)C1
  smiles  Euclidean_metric
0                     30.0
--------------------------------------------------------------------------------
CNC(=O)C1=CC=C(C)C(NC(=O)NC(C)C2=CC=CC=C2OCC3=CC=CC=C3)=C1
  smiles  Euclidean_metric
0                28.071338
--------------------------------------------------------------------------------
COC1(C)CCN(C(=O)C2=CC=CC=C2SC(F)(F)F)CC1
  smiles  Euclidean_metric
0                25.059928
--------------------------------------------------------------------------------
CS(=O)(=O)CCC1NC(=O)N(CCOC2=CC=CC(Cl)=C2)C1=O
  smiles  Euclidean_metric
0                26.172505
--------------------------------------------------------------------------------


## **Test the model for representation**

In [14]:
import selfies as sf
from rdkit import Chem

smiles2selfie_fn = lambda x: sf.encoder(x)
selfie2smiles_fn = lambda x: sf.decoder(x)
canonical_fn = lambda smi: Chem.MolToSmiles(Chem.MolFromSmiles(smi), isomericSmiles=False)
smiles_clean_fn = lambda x: ('[' not in x) and ('B' not in x.replace('Br', '')) and ('.' not in x)
len_filter_fn = lambda x: sf.len_selfies(x)<=selfies_max_len

#### **Preprocess drugs**

In [15]:
from data_utils import _check_validity_smiles

df = pd.read_csv('/home/rkmvu/Dataset/selfies/drug/mdrug.csv')
_, valid_mask = _check_validity_smiles(smiles=df['smiles'].tolist())
df = df[valid_mask].copy()
df['smiles'] = df['smiles'].apply(canonical_fn)
df['selfies'] = df['smiles'].apply(smiles2selfie_fn)

len_mask = df['selfies'].apply(len_filter_fn)
df = df[len_mask]

df_temp = df.copy()
mask = df_temp['smiles'].apply(smiles_clean_fn)
df_temp = df_temp[mask]
df_temp = df_temp.drop_duplicates(subset=['smiles'])
selfies_main = df_temp['selfies'].tolist()
print(f'Number of selfies: {len(selfies_main)}')
df_temp

<===checking smiles validity===>
----------------------------------------------------------------------


100%|██████████| 1381/1381 [00:00<00:00, 4720.96it/s]


Number of selfies: 1089


,name,smiles,InChl,type,selfies
0,Abacavir,Nc1nc(NC2CC2)c2ncn(C3C=CC(CO)C3)c2n1,InChI=1S/C14H18N6O/c15-14-18-12(17-9-2-3-9)11-...,Drug,[N][C][=N][C][Branch1][#Branch1][N][C][C][C][R...
1,Abiraterone,CC(=O)OC1CCC2(C)C(=CCC3C2CCC2(C)C(c4cccnc4)=CC...,InChI=1S/C26H33NO2/c1-17(28)29-20-10-12-25(2)1...,Drug,[C][C][=Branch1][C][=O][O][C][C][C][C][Branch1...
2,Acamprosate,CC(=O)NCCCS(=O)(=O)O,"InChI=1S/C5H11NO4S/c1-5(7)6-3-2-4-11(8,9)10/h2...",Drug,[C][C][=Branch1][C][=O][N][C][C][C][S][=Branch...
3,Acarbose,CC1OC(OC2C(CO)OC(OC3C(CO)OC(O)C(O)C3O)C(O)C2O)...,InChI=1S/C25H43NO18/c1-6-11(26-8-2-7(3-27)12(3...,Drug,[C][C][O][C][Branch2][Ring2][#Branch2][O][C][C...
4,Acebutolol,CCCC(=O)Nc1ccc(OCC(O)CNC(C)C)c(C(C)=O)c1,InChI=1S/C18H28N2O4/c1-5-6-18(23)20-14-7-8-17(...,Drug,[C][C][C][C][=Branch1][C][=O][N][C][=C][C][=C]...
...,...,...,...,...,...
1374,Ziprasidone,O=C1Cc2cc(CCN3CCN(c4nsc5ccccc45)CC3)c(Cl)cc2N1,InChI=1S/C21H21ClN4OS/c22-17-13-18-15(12-20(27...,Drug,[O][=C][C][C][=C][C][Branch2][Ring1][#Branch2]...
1375,Zoledronate,O=P(O)(O)C(O)(Cn1ccnc1)P(=O)(O)O,"InChI=1S/C5H10N2O7P2/c8-5(15(9,10)11,16(12,13)...",Drug,[O][=P][Branch1][C][O][Branch1][C][O][C][Branc...
1377,Zolpidem,Cc1ccc(-c2nc3ccc(C)cn3c2CC(=O)N(C)C)cc1,InChI=1S/C19H21N3O/c1-13-5-8-15(9-6-13)19-16(1...,Drug,[C][C][=C][C][=C][Branch2][Ring1][O][C][N][=C]...
1378,Zonisamide,NS(=O)(=O)Cc1noc2ccccc12,"InChI=1S/C8H8N2O3S/c9-14(11,12)5-7-6-3-1-2-4-8...",Drug,[N][S][=Branch1][C][=O][=Branch1][C][=O][C][C]...


### **Finding Nearest Neighbours**

In [16]:
from sklearn.neighbors import NearestNeighbors

z = encode(smiles_or_selfies=selfies_main)
n_neigh = NearestNeighbors(n_neighbors=30, metric='euclidean')
n_neigh.fit(z)

<===doing selfies reconstruction===>
----------------------------------------------------------------------


100%|██████████| 3/3 [00:00<00:00, 13.96it/s]


NearestNeighbors(metric='euclidean', n_neighbors=30)

In [17]:
distances, indices = n_neigh.kneighbors(z)
distances
z.shape

(1089, 40)

In [18]:
name2selfies('Abacavir')

'[N][C][=N][C][Branch1][#Branch1][N][C][C][C][Ring1][Ring1][=C][N][=C][N][Branch1][N][C][C][=C][C][Branch1][Ring1][C][O][C][Ring1][#Branch1][C][Ring1][N][=N][Ring2][Ring1][Ring2]'

In [19]:
selfies2name(name2selfies('Abacavir'))

'Abacavir'

In [20]:
smi2nneigh(name2selfies('Loxapine'),indices=indices)

{'selfies': ['[C][O][C][=C][C][=N][C][Branch2][Ring1][=Branch1][N][C][C][N][Branch1][N][C][=Branch1][C][=O][C][C][C][C][O][Ring1][Branch1][C][C][Ring1][=N][=N][C][Branch1][C][N][=C][Ring2][Ring1][Ring2][C][=C][Ring2][Ring1][Branch2][O][C]',
  '[C][C][C][=Branch1][C][=O][C][=C][C][=C][C][=Branch1][Ring2][=C][Ring1][=Branch1][N][Branch1][S][C][C][N][C][C][N][Branch1][Ring2][C][C][O][C][C][Ring1][=Branch2][C][=C][C][=C][C][=C][Ring1][=Branch1][S][Ring2][Ring1][=Branch1]',
  '[C][O][C][C][C][Branch2][#Branch1][Ring1][O][C][C][Branch1][C][C][C][=Branch1][C][=O][O][C][Branch1][C][C][C][Branch1][C][C][C][Branch1][#Branch1][O][C][Branch1][C][C][=O][C][Branch1][C][C][C][=Branch1][C][=O][C][Branch1][Branch1][C][O][Ring1][Ring1][C][C][Branch1][C][C][C][Branch2][Ring1][#Branch2][O][C][O][C][Branch1][C][C][C][C][Branch1][=Branch1][N][Branch1][C][C][C][C][Ring1][#Branch2][O][C][Branch1][C][C][=O][C][Ring2][Ring2][#Branch2][C][O][C][Branch1][C][C][C][Ring2][Branch1][Ring1][O][C][Branch1][C][C][=O]',


## **Nearest Neighbour Analysis**

In [21]:
import numpy as np

top_drugs = ["Metformin", "Amoxicillin", "Atorvastatin", "Amlodipine", "Acetaminophen", "Imatinib", "Clozapine", 
             "Ibuprofen", "Azithromycin", "Doxycycline"]
top_drugs = df_temp['name']
n_neigh=10

all_out = {'drug_evslfi':[], 'nn_idx_evslfi':[], 'smiles_evslfi':[], 'names_evslfi':[]}
for drug in top_drugs:
    _smiles = name2selfies(drug)
    out = smi2nneigh(_smiles, indices=indices, n_neigh=n_neigh)
    drug_name = [drug]*n_neigh
    idx = np.arange(1, 11)
    out = {'drug':drug_name, 'nn_idx':idx, **out}
    all_out['drug_evslfi'].extend(out['drug'])
    all_out['nn_idx_evslfi'].extend(out['nn_idx'])
    all_out['smiles_evslfi'].extend(map(selfie2smiles_fn, out['selfies']))
    all_out['names_evslfi'].extend(out['name'])

In [22]:
all_out_df = pd.DataFrame(all_out)
all_out_df
# all_out_df.to_csv('/home/rkmvu/Codes/ICDCIT_submission/results/near_smiles_evslfi.csv', index=False)

,drug_evslfi,nn_idx_evslfi,smiles_evslfi,names_evslfi
0,Abacavir,1,CC(C=CC1=C(C)CCCC1(C)C)=CC=CC(C)=CC(=O)O,Alitretinoin
1,Abacavir,2,CC1CC2C3CCC4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C1(O...,Clobetasol
2,Abacavir,3,CC(O)C1C(=O)N2C(C(=O)O)=C(SC3CNC(C(=O)N(C)C)C3...,Meropenem
3,Abacavir,4,CCCCNC(=O)NS(=O)(=O)C1=CC=C(C)C=C1,Tolbutamide
4,Abacavir,5,CC1=CC(C(C)(C)C)=C(O)C(C)=C1CC2=NCCN2,Oxymetazoline
...,...,...,...,...
10885,Zuclopenthixol,6,COC1=CC=C2C3=C1OC4C(O)C=CC5C(C2)N(C)CCC354,Codeine
10886,Zuclopenthixol,7,COC1C(OC(C)=O)CC(=O)OC(C)CC=CC=CC(O)C(C)CC(CC=...,Josamycin
10887,Zuclopenthixol,8,CCCCCC(O)C=CC1C(O)CC(=O)C1CCCCCCC(=O)O,Alprostadil
10888,Zuclopenthixol,9,CC12C=CC(=O)NC1CCC3C2CCC4(C)C(C(=O)NC5=CC(C(F)...,Dutasteride


### **Reconstruction accuracy**

In [23]:
z = encode(smiles_or_selfies=selfies_main)
selfies_recon = decode(x_latent=z)
print(z.shape)

exact = [x==y for x, y in zip(selfies_main, selfies_recon)]
smis, checks,_ = _check_validity_selfies(selfiles=selfies_recon)

print('='*80)
print(f'Exact reconstruction: {sum(exact)/z.shape[0]}')
print(f'Valid reconstruction: {sum(checks)/z.shape[0]}')
print('='*80)


<===doing selfies reconstruction===>
----------------------------------------------------------------------


100%|██████████| 3/3 [00:00<00:00, 20.34it/s]


<===doing selfies reconstruction===>
----------------------------------------------------------------------


  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 18.28it/s]


(1089, 40)
<===checking selfies validity===>
----------------------------------------------------------------------


100%|██████████| 1089/1089 [00:00<00:00, 7861.10it/s]

Exact reconstruction: 0.0
Valid reconstruction: 1.0
